In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

import pickle

In [4]:
data = pd.read_csv("../Data/survey-data.csv")

print("Shape:", data.shape)
data.head()


Shape: (52, 34)


,Timestamp,Do you agree to participate voluntarily in this research survey?,Age Group,Gender,Field of Study (Optional),Current Occupation / Status,[Felt upset because of something unexpected],[Felt unable to control important things in life],[Felt nervous and stressed],[Felt confident about handling personal problems],...,[I make mistakes because my mind is on other things],[I feel unable to control my emotions],[I do not recognize myself in the way I react emotionally],[I may overreact unintentionally],Average sleep duration per night,Daily screen time (non-academic/work),How often do you take breaks during study/work?,Describe how you have been feeling mentally and emotionally over the past few weeks.,What are the main factors currently causing you stress or mental exhaustion?,Would you like access to mental health support resources?
0,2026/02/04 12:55:30 pm GMT+5:30,I Agree,19-22,Male,NaN,College Student,Fairly Often,Very Often,Very Often,Fairly Often,...,Often,Often,Often,Often,5-6 hours,2–4 hours,Sometimes,Not so good,A person,No
1,2026/02/04 1:19:21 pm GMT+5:30,I Agree,19-22,Male,Cse,College Student,Fairly Often,Sometimes,Fairly Often,Sometimes,...,Rarely,Rarely,Rarely,Never,5-6 hours,More than 6 hours,Regularly,Exhausted while thinking about my future,Career and relationships,Yes
2,2026/02/04 1:55:12 pm GMT+5:30,I Agree,19-22,Male,Bfa,College Student,Sometimes,Sometimes,Almost Never,Fairly Often,...,Sometimes,Sometimes,Often,Sometimes,5-6 hours,More than 6 hours,Rarely,Fully stressed and overthinking,Stress,Yes
3,2026/02/04 4:08:08 pm GMT+5:30,I Agree,19-22,Male,NaN,College Student,Fairly Often,Fairly Often,Fairly Often,Never,...,Never,Never,Rarely,Never,5-6 hours,Less than 2 hours,Sometimes,Sometimes I do feel I have been ushered to mur...,Perhaps I have never comprehend my emotions,No
4,2026/02/04 7:11:06 pm GMT+5:30,I Agree,19-22,Male,NaN,College Student,Very Often,Fairly Often,Sometimes,Sometimes,...,Never,Rarely,Sometimes,Rarely,5-6 hours,2–4 hours,Sometimes,Good,Money,Yes


In [6]:
data = data.drop(columns=[
"Timestamp",
"Do you agree to participate voluntarily in this research survey?",
"Field of Study (Optional)",
"Describe how you have been feeling mentally and emotionally over the past few weeks.",
"What are the main factors currently causing you stress or mental exhaustion?",
"Would you like access to mental health support resources?"
], errors="ignore")

In [8]:
mapping = {
"Never":0,
"Rarely":1,
"Sometimes":2,
"Often":3,
"Always":4,
"Almost Never":1,
"Fairly Often":3,
"Very Often":4
}

data.replace(mapping, inplace=True)

/var/folders/k_/s80f4rtj5f99d7llylsvsf4m0000gn/T/ipykernel_3330/1021105837.py:12: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data.replace(mapping, inplace=True)


In [10]:
#Age
data["Age Group"] = data["Age Group"].map({
"15-18":0,
"19-22":1,
"23-30":2,
"31-45":3
})

#Gender
data["Gender"] = data["Gender"].map({
"Male":0,
"Female":1,
"Prefer not to say":2
})

#Occupation
data["Current Occupation / Status"] = data["Current Occupation / Status"].map({
"School student":0,
"College student":1,
"Working professional":2,
"Other":3
})

#Sleep
data["Average sleep duration per night"] = data["Average sleep duration per night"].map({
"Less than 5 hours":0,
"5–6 hours":1,
"6–7 hours":2,
"More than 7 hours":3
})

#Screen Time
data["Daily screen time (non-academic/work)"] = data["Daily screen time (non-academic/work)"].map({
"Less than 2 hours":0,
"2–4 hours":1,
"4–6 hours":2,
"More than 6 hours":3
})

#Break 
data["How often do you take breaks during study/work? "] = data["How often do you take breaks during study/work? "].map({
"Rarely":0,
"Sometimes":1,
"Often":2,
"Always":3
})


In [12]:
#PSS_Columns
pss_cols = [
" [Felt upset because of something unexpected]",
" [Felt unable to control important things in life]",
" [Felt nervous and stressed]",
" [Felt confident about handling personal problems]",
" [Felt things were going your way]",
" [Could not cope with all things you had to do]",
" [Able to control irritations]",
" [Felt on top of things]",
" [Angered by things outside your control]",
" [Felt difficulties piling up]"
]

In [14]:
reverse_cols = [
" [Felt confident about handling personal problems]",
" [Felt things were going your way]",
" [Able to control irritations]",
" [Felt on top of things]"
]

for col in reverse_cols:
    data[col] = 4 - data[col]


In [16]:
#Stress Score
data["stress_score"] = data[pss_cols].sum(axis=1)

#Stress Level
def stress_label(score):
    if score <= 13:
        return 0
    elif score <= 26:
        return 1
    else:
        return 2

data["stress_level"] = data["stress_score"].apply(stress_label)

In [18]:
#Missing_values
data = data.fillna(data.median(numeric_only=True))


In [20]:
X = data.drop(columns=["stress_score","stress_level"])
y = data["stress_level"]

print(X.shape)

(52, 28)


In [22]:
#Top_Feature_Selection
rf = RandomForestClassifier(random_state=42)
rf.fit(X, y)

importances = rf.feature_importances_

feature_importance = pd.DataFrame({
"feature":X.columns,
"importance":importances
}).sort_values(by="importance",ascending=False)

feature_importance

,feature,importance
9,[Able to control irritations],0.086717
12,[Felt difficulties piling up],0.075482
4,[Felt unable to control important things in l...,0.059800
8,[Could not cope with all things you had to do],0.058327
21,[I make mistakes because my mind is on other ...,0.057170
23,[I do not recognize myself in the way I react...,0.057159
7,[Felt things were going your way],0.046701
26,Daily screen time (non-academic/work),0.044711
15,[I feel physically exhausted],0.044415
13,[I feel mentally exhausted],0.041824


In [24]:
top_features = feature_importance["feature"].head(12)

X = X[top_features]

print("New shape:", X.shape)

New shape: (52, 12)


In [26]:
#Logistic_Regression Baseline Model

X_train, X_test, y_train, y_test = train_test_split(
X, y, test_size=0.2, random_state=42
)
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test,pred))
print(classification_report(y_test,pred))


Accuracy: 0.8181818181818182
              precision    recall  f1-score   support

           0       1.00      0.50      0.67         2
           1       0.78      1.00      0.88         7
           2       1.00      0.50      0.67         2

    accuracy                           0.82        11
   macro avg       0.93      0.67      0.74        11
weighted avg       0.86      0.82      0.80        11



In [28]:
scores = cross_val_score(model,X,y,cv=5)

print("Cross-validation scores:",scores)
print("Mean accuracy:",scores.mean())


Cross-validation scores: [0.72727273 0.63636364 0.8        0.8        0.9       ]
Mean accuracy: 0.7727272727272727


In [34]:
#Save Model 
import os
import pickle

os.makedirs("../Model", exist_ok=True)

pickle.dump(model, open("../Model/stress_model.pkl","wb"))

In [36]:
import os
os.listdir("../Model")

['stress_model.pkl']